# Turbine blade defect detector — v2

Retrain after the v1 audit. Every setting below that differs from v1 is there for a reason
recorded in `docs/DATASET_AUDIT_v1.md`; the comments say which.

**Before you start**, upload this repo (or at least `train/`, `valid/`, `test/`, `tools/`,
`training/`) as a Kaggle Dataset, and attach it to this notebook. Set the accelerator to
**GPU P100** — not T4 x2. The v1 run used `device=0,1`, and DDP synchronisation across two
T4s at this batch size costs more than the second GPU returns.

Expect roughly 2-3 hours for 50 epochs, comfortably inside Kaggle's 9-hour session limit.
The v1 run asked for 100 epochs, converged at 38, and died to a timeout at 58.

In [ ]:
!pip install -q ultralytics onnx onnxruntime
import torch, ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)

if not torch.cuda.is_available():
    raise SystemExit(
        "\n" + "=" * 74 +
        "\nNO GPU. Stop here - do not run the training cell, it will fail or take days."
        "\n" + "=" * 74 +
        "\n"
        "\nKaggle gave this session a CPU-only image because no accelerator is selected."
        "\n"
        "\nFIX:"
        "\n  1. Click 'Edit' (top right) if you are in viewer mode - the settings panel"
        "\n     only exists in the editor."
        "\n  2. Open the right-hand panel: click the '<' arrow on the right edge, or"
        "\n     the three-dot menu -> Session options."
        "\n  3. Accelerator: change 'None' to 'GPU P100'."
        "\n     (P100, not T4 x2 - DDP across two T4s costs more than the second GPU"
        "\n      returns at this batch size.)"
        "\n  4. Accept the restart. It WIPES /kaggle/working, so then click 'Run All'."
        "\n"
        "\nIf Accelerator is greyed out, Kaggle requires phone verification for GPU:"
        "\n  kaggle.com -> Settings -> Phone Verification."
        "\n"
        "\nInternet must also be On in the same panel (needed to clone the repo)."
        "\n"
    )

print("cuda:", torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print(f"vram: {total / 1e9:.1f} GB total, {free / 1e9:.1f} GB free")
if total < 14e9:
    print("\n! Under 14 GB. Drop batch to 4 in the training cell if you hit OOM.")


## 1. Rebuild the dataset

Do not train on the raw export. `rebuild_turbine.py` drops `healthy` as a class, keeps those
images as background negatives, re-splits on contiguous capture-ID blocks to remove
near-duplicate leakage, and normalises the polygon labels to boxes.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

# Locate the repo however you got it onto Kaggle. Three routes all work:
#   1. Attach it as a Kaggle Dataset  (Add Input -> Datasets -> your upload)
#   2. Kaggle can build a Dataset straight from a GitHub URL (New Dataset -> Link -> GitHub)
#   3. Nothing attached? This clones it, which needs Internet switched ON in the
#      notebook settings sidebar. Kaggle disables internet by default.
REPO_URL = "https://github.com/abyyworld/Drone-visualisation-training.git"
BRANCH   = "claude/model-retrain-solar-panels-n77z3v"
WORK     = Path("/kaggle/working/drone-inspection")

def looks_like_repo(p: Path) -> bool:
    return (p / "tools" / "rebuild_turbine.py").exists() and (p / "train" / "images").is_dir()

inputs = sorted(Path("/kaggle/input").glob("*")) if Path("/kaggle/input").exists() else []
print("Attached inputs:", [p.name for p in inputs] or "(none)")

source = None
for entry in inputs:
    if not entry.is_dir():
        continue
    # The repo may be the input itself, or one level down if the upload was zipped.
    for candidate in [entry, *[d for d in entry.iterdir() if d.is_dir()]]:
        if looks_like_repo(candidate):
            source = candidate
            break
    if source:
        break

if source:
    print(f"Found the repo at {source}")
    if not WORK.exists():
        shutil.copytree(source, WORK)
else:
    print("No attached input looks like the repo - cloning from GitHub.")
    print("If this fails, turn Internet ON in the notebook settings sidebar.")
    if not WORK.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        REPO_URL, str(WORK)], check=True)

os.chdir(WORK)
sys.path.insert(0, str(WORK))

assert looks_like_repo(WORK), (
    f"Repo looks incomplete at {WORK}. Expected tools/ and train/images/. "
    f"Contents: {[p.name for p in WORK.iterdir()][:20]}"
)
print(f"\nOK -> {WORK}")
print("train images:", len(list((WORK / 'train' / 'images').iterdir())))

# Rebuild into a leakage-free, defect-only dataset. --min-sharpness 20 drops the ~15% of
# annotated images that are smears, where a box cannot teach localisation.
!python3 tools/rebuild_turbine.py --out /kaggle/working/turbine_v2 --copy --min-sharpness 20


In [ ]:
# The rebuild exists to make these checks pass. If any fail, fix the data before training -
# training on a dataset that fails this audit is how v1 happened.
!python3 tools/audit_dataset.py /kaggle/working/turbine_v2

## 2. Train

| Setting | v1 | v2 | Why |
|---|---|---|---|
| model | `yolov8m` (25.9M) | `yolo11s` (9.4M) | Smaller + higher resolution beats bigger + low resolution for small defects, and trains faster. It also has to fit in a browser download. |
| `imgsz` | 640 | **960** | 21.6% of `surface_peeling` boxes are under 1% of frame area. Resolution is the dominant lever for small objects. |
| `epochs` | 100 | **50**, `patience=10` | v1 peaked at epoch 38 and was flat from ~30. |
| `optimizer` | `auto` | explicit `SGD` | `auto` silently chose MuSGD and drove `lr0` to 0.029 during warmup, collapsing mAP50 from 0.411 to 0.183 at epoch 3. |
| `warmup_epochs` | 3 | **5** + `cos_lr` | Smooths out that same spike. |
| `cache` | `True` (7.5 GB RAM) | `disk` | Ultralytics flags RAM caching as non-deterministic even with `seed=0`. |
| `device` | `0,1` | **single GPU** | DDP overhead exceeded the benefit at this scale. |

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

results = model.train(
    data="/kaggle/working/turbine_v2/data.yaml",
    epochs=50,
    imgsz=960,
    batch=8,               # 960px on a 16GB P100. Drop to 4 if you hit OOM.
    patience=10,

    optimizer="SGD",       # explicit, so it cannot be silently overridden
    lr0=0.01, lrf=0.01, momentum=0.937, weight_decay=0.0005,
    warmup_epochs=5, cos_lr=True,

    cache="disk",
    device=0,
    workers=2,
    seed=0, deterministic=True,

    # Defects are small and orientation-varied; blades appear at any angle from a drone.
    degrees=10.0, fliplr=0.5, flipud=0.2, scale=0.5,
    mosaic=1.0, close_mosaic=10,

    save_period=10,        # survives a session dying mid-run
    project="/kaggle/working/runs", name="turbine_v2", exist_ok=True,
    plots=True,
)

## 3. Evaluate honestly

Aggregate mAP is not the number to quote. v1 scored 0.782 aggregate while being useless in
the field, because `healthy` was 62% of instances and trivially separable. v2 has no such
free class, so **expect a lower aggregate — around 0.45-0.60 — and treat that as progress.**

Judge per-class defect AP, and the visual check in section 4.

In [ ]:
!python3 tools/evaluate.py /kaggle/working/runs/turbine_v2/weights/best.pt \
    --data /kaggle/working/turbine_v2/data.yaml --split test --imgsz 960 \
    --out /kaggle/working/runs/eval_v2

## 4. The check that actually matters

Metrics on a split drawn from the same pool as training cannot tell you whether the model
works on a photo unlike anything it trained on. v1 never had this run against it.

Attach a folder of real turbine photos from any unrelated source (search results, a public
dataset, your own flight) and **look at the boxes**. A model that boxes sky, ground, or the
blade edge is failing in a way no in-distribution metric will reveal.

In [ ]:
REALITY_CHECK = Path("/kaggle/input/turbine-reality-check")   # <- your own folder

if REALITY_CHECK.exists():
    !python3 tools/evaluate.py /kaggle/working/runs/turbine_v2/weights/best.pt \
        --predict {REALITY_CHECK} --imgsz 960 --conf 0.25 \
        --out /kaggle/working/runs/reality
else:
    print("No reality-check folder attached. Do this before trusting the model.")

## 5. Export for the web app

int8 quantisation takes the download from ~38 MB to ~10 MB but usually costs a couple of
points of mAP. Measure it, then decide.

In [ ]:
!python3 tools/export_onnx.py /kaggle/working/runs/turbine_v2/weights/best.pt \
    --name turbine --imgsz 960

# What did quantisation cost?
!python3 tools/evaluate.py web/models/turbine.onnx \
    --data /kaggle/working/turbine_v2/data.yaml --split test --imgsz 960 \
    --out /kaggle/working/runs/eval_onnx

In [ ]:
# Download web/models/turbine.onnx and manifest.json from the notebook output,
# commit them to web/models/, and the Pages deploy picks them up automatically.
!ls -lh web/models/